# Silver Layer - Data Cleanup

**SariMart Express - Automated Replenishment Demo (Microsoft Fabric)**

Reads every **Bronze** table, applies a simple, **schema-preserving** cleanup, and writes the
conformed `silver_<table>` Delta tables. Runs across all 9 tables (7 dimensions + 2 facts).

**Cleanup applied (none of these change the schema):**
1. **Trim whitespace** on string columns (type stays STRING).
2. **Drop fully-duplicate rows**.
3. **Enforce key integrity** - remove rows with a null primary key and any duplicate primary key.

**Schema guarantee:** before each write, a guard compares the cleaned DataFrame's
(column name + data type) list against the Bronze source. If they differ, the cell **fails fast** -
so the Silver schema can never drift from Bronze. No columns are added, dropped, renamed, or recast.

> **Prerequisite:** run the **Bronze** notebook first. Attach `LH_Retail`, then **Run all**.

In [ ]:
# 1. Configuration - every table and its primary key
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

BRONZE_SCHEMA = "bronzedb"
SILVER_SCHEMA = "silverdb"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")

# table -> primary key (used for null-key and duplicate-key removal)
TABLES = {
    "date_dim":                "date_key",
    "store_dim":               "store_key",
    "category_dim":            "category_key",
    "supplier_dim":            "supplier_key",
    "product_dim":             "product_key",
    "promotion_dim":           "promotion_key",
    "channel_dim":             "channel_key",
    "sales_dim":               "sales_key",
    "inventory_snapshot_dim":  "inventory_key",
}
print(f"{len(TABLES)} tables to clean from {BRONZE_SCHEMA} into {SILVER_SCHEMA}")

In [ ]:
# 2. Cleanup function - schema-preserving
def clean(table, pk):
    df0 = spark.table(f"{BRONZE_SCHEMA}.{table}")
    before = df0.count()

    # schema contract = (name, type) for every column, captured from Bronze
    contract = [(f.name, f.dataType.simpleString()) for f in df0.schema.fields]

    # 1. trim whitespace on STRING columns only (preserves the column type)
    df = df0
    for fld in df0.schema.fields:
        if isinstance(fld.dataType, StringType):
            df = df.withColumn(fld.name, F.trim(F.col(fld.name)))

    # 2. drop fully-duplicate rows  3. enforce key integrity (no null / dup PK)
    df = df.dropDuplicates().dropna(subset=[pk]).dropDuplicates([pk])

    # 4. SCHEMA GUARD - cleaned schema must equal the Bronze contract exactly
    after = [(f.name, f.dataType.simpleString()) for f in df.schema.fields]
    assert after == contract, f"SCHEMA DRIFT in {table}:\n  bronze={contract}\n  silver={after}"

    target = f"{SILVER_SCHEMA}.{table}"
    df.write.mode("overwrite").format("delta").saveAsTable(target)
    after_rows = spark.table(target).count()
    return before, after_rows

In [ ]:
# 3. Run cleanup across all tables
print(f"{'table':42s}{'bronze':>10s}{'silver':>10s}{'removed':>10s}")
print("-" * 72)
for t, pk in TABLES.items():
    before, after = clean(t, pk)
    print(f"{(SILVER_SCHEMA+'.'+t):42s}{before:>10,}{after:>10,}{(before-after):>10,}")
print(f"\n✓ all silver tables written to {SILVER_SCHEMA} with schema preserved")

In [ ]:
# 4. Validation - confirm every silver schema matches its bronze source
print("Schema parity check (bronzedb vs silverdb):\n")
all_ok = True
for t in TABLES:
    b = [(f.name, f.dataType.simpleString()) for f in spark.table(f"{BRONZE_SCHEMA}.{t}").schema.fields]
    s = [(f.name, f.dataType.simpleString()) for f in spark.table(f"{SILVER_SCHEMA}.{t}").schema.fields]
    ok = (b == s)
    all_ok &= ok
    print(f"  {(SILVER_SCHEMA+'.'+t):42s} {'PASS' if ok else 'FAIL'}  ({len(s)} cols)")
print("\nAll schemas preserved:", all_ok)

## Notes

- **Schema-preserving by design:** the only transformations are whitespace trimming, full-row
  de-duplication, and primary-key integrity. No columns are added, dropped, renamed, or recast,
  and the schema guard asserts this before every write.
- **Idempotent:** `mode("overwrite")` means re-runs fully refresh the silver tables; on the synthetic
  data (already clean and unique) the `removed` count will typically be 0 - which is the expected,
  healthy result.
- **Audit columns intentionally omitted:** adding ingestion timestamps would change the schema, so
  they are left out to honour the "schema will not change" requirement. Track lineage at the table
  level (Delta history) instead.
- **Downstream:** the Replenishment Engine can be pointed at these cleaned `silver_<table>` tables
  by swapping its `bronze_` prefixes for `silver_`.